In [ ]:
# hide warnings
import warnings
warnings.filterwarnings('ignore')

# https://groups.csail.mit.edu/sls/downloads/restaurant/


In [ ]:
!pip install -U transformers
!pip install -U accelerate
!pip install -U datasets

In [ ]:
import pandas as pd
import json
import requests


In [ ]:
train = pd.read_csv("https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/master/mit_restaurant_search_ner/train.bio", sep="\t", header=None)
train.head()


In [ ]:
response = requests.get("https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/master/mit_restaurant_search_ner/train.bio")
response = response.text

In [ ]:
response = response.splitlines()

In [ ]:
response[:]

In [ ]:
train_tokens = []
train_tags = []

temp_tags = []
temp_tokens = []

for line in response:
  if line != "":
    tag,token = line.strip().split("\t")
    temp_tags.append(tag)
    temp_tokens.append(token)
  else:
    train_tokens.append(temp_tokens)
    train_tags.append(temp_tags)
    temp_tokens,temp_tags = [],[]


In [ ]:
train_tags

In [ ]:
len(train_tokens),len(train_tags)

In [ ]:
response = requests.get("https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/master/mit_restaurant_search_ner/test.bio")
response = response.text
response = response.splitlines()
test_tokens = []
test_tags = []

temp_tags = []
temp_tokens = []

for line in response:
  if line != "":
    tag,token = line.strip().split("\t")
    temp_tags.append(tag)
    temp_tokens.append(token)
  else:
    test_tokens.append(temp_tokens)
    test_tags.append(temp_tags)
    temp_tokens,temp_tags = [],[]

len(test_tokens),len(test_tags)


HUGGING face data prep


In [ ]:
from datasets import Dataset,DatasetDict

df = pd.DataFrame(data={"tokens":train_tokens,"ner_tags_str":train_tags})
train = Dataset.from_pandas(df)

df = pd.DataFrame(data={"tokens":test_tokens,"ner_tags_str":test_tags})
test = Dataset.from_pandas(df)


dataset = DatasetDict({'train':train,'test':test,'validation':test})
dataset

In [ ]:
dataset

In [ ]:
dataset['train'][0]

In [ ]:
unique_tags = set()
for tag in dataset['train']['ner_tags_str']:
    unique_tags.update(tag)
unique_tags = list(set([x[2:] for x in list(unique_tags) if x!='O']))
tag2index = {"O": 0}
for i, tag in enumerate(unique_tags):
    tag2index[f'B-{tag}'] = len(tag2index)
    tag2index[f'I-{tag}'] = len(tag2index)

index2tag = {v:k for k,v in tag2index.items()}

In [ ]:
tag2index,index2tag

In [ ]:
dataset = dataset.map(lambda example: {"ner_tags": [tag2index[tag] for tag in example['ner_tags_str']]})

In [ ]:
dataset

MODEL BUILDING

In [ ]:
from transformers import AutoTokenizer

In [ ]:
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

In [ ]:
dataset['train'][2]

In [ ]:
input = dataset['train'][2]['tokens']
output = tokenizer(input, is_split_into_words=True)
tokenizer.convert_ids_to_tokens(output.input_ids)

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples['tokens'], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # if id=-100 then loss is not calculated
            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs['labels'] = labels

    return tokenized_inputs



In [ ]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

In [ ]:
tokenized_dataset['train'][2]

In [ ]:
dataset['train'][2]

## Data Collation and Metrics


In [ ]:
!pip install seqeval
!pip install evaluate

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
import evaluate
import numpy as np

metric = evaluate.load('seqeval')
label_names = list(tag2index)

def compute_metrics(eval_preds):
    logits, labels = eval_preds

    predictions = np.argmax(logits, axis=-1)
    true_labels = [[label_names[l] for l in label if l!=-100] for label in labels]

    true_predictions = [[label_names[p] for p, l in zip(prediction, label) if l != -100]
                        for prediction, label in zip(predictions, labels)]

    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": all_metrics['overall_precision'],
        'recall': all_metrics['overall_recall'],
        'f1': all_metrics['overall_f1'],
        'accuracy': all_metrics['overall_accuracy'],
    }

MODEL TRAINING

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(model_ckpt, id2label=index2tag, label2id=tag2index)

In [ ]:
from transformers import TrainingArguments, Trainer


In [ ]:
args = TrainingArguments("finetuned-ner", eval_strategy='epoch',
                         save_strategy='epoch',
                         learning_rate=2e-5,
                         num_train_epochs=10,
                         weight_decay=0.01)

In [ ]:
trainer = Trainer(model=model, args=args,
                  train_dataset=tokenized_dataset['train'],
                  eval_dataset=tokenized_dataset['validation'],
                  data_collator=data_collator,
                  compute_metrics=compute_metrics)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("ner_distilbert")

In [ ]:
from transformers import pipeline

checkpoint = "ner_distilbert"
pipe = pipeline('token-classification', model=checkpoint, aggregation_strategy='simple')

In [ ]:
pipe("which restaurant serves the best shushi in new york?")